In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import setup_plotting, switch_cwd_to_root

switch_cwd_to_root()

from spatial_tcr.pl import utils as plot_utils

figure_dir = "figures/revision/figure-4"
figure_dir_2 = "figures/revision/supplement"
setup_plotting(figure_dir, display_dpi=300, save_dpi=300)
setup_plotting(figure_dir_2, display_dpi=300, save_dpi=300)

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from tqdm.auto import tqdm

In [ ]:
path = "data/xenium/processed/05.2-kidney_tcr_infilrate.h5ad"
adata = sc.read_h5ad(path)
adata

In [ ]:
# Create a mapping of sample names to numbers
sample_map = {
    name: f"Sample {i + 1}" for i, name in enumerate(adata.obs["sample"].unique())
}

# Create a copy of the data and map sample names
plot_data = (
    adata.obs.groupby(["condition", "sample"], observed=True)["cell_type_l2"]
    .value_counts()
    .unstack()["gdT"]
    .copy()
)
plot_data.index = plot_data.index.set_levels(
    [plot_data.index.levels[0], plot_data.index.levels[1].map(sample_map)]
)

# Plot with colors by condition
colors = {"ANCA": "red", "Control": "blue"}
sns.set_theme(context="paper", style="ticks")
fig, ax = plt.subplots(figsize=(6, 4))
ax = plot_data.plot(
    kind="bar",
    color=[colors[c] for c in plot_data.index.get_level_values("condition")],
    ax=ax,
)
ax.set_ylabel("Number of gdT cells")
ax.set_xlabel("Sample ")
ax.set_title("gdT cells in kidney samples")
# Add legend with condition colors
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in colors.values()]
ax.legend(handles, colors.keys(), title="Condition")

sns.despine()

In [ ]:
adata = adata[adata.obs["condition"] == "ANCA"].copy()

In [ ]:
df = (
    adata.obs.groupby("tcell_infiltrate", observed=True)["cell_type_l2"]
    .value_counts()
    .unstack()
)
# Column normalize the dataframe
df_norm = df.div(df.sum(axis=0), axis=1)

gdt_distrib = df_norm["gdT"]

In [ ]:
df = (
    adata.obs.groupby("tcell_infiltrate", observed=True)["cell_type_l1"]
    .value_counts()
    .unstack()
)
df_norm = df.div(df.sum(axis=0), axis=1) * 100

t_distrib = df_norm["T"]

In [ ]:
df = (
    adata.obs.groupby("tcell_infiltrate", observed=True)["cell_type_l2"]
    .value_counts(normalize=True)
    .unstack()
)
df

## calc statistic

In [ ]:
ad_t = adata[adata.obs["cell_type_l1"] == "T"].copy()

In [ ]:
palette = {
    "outside infiltrate": "#c5c5c5",
    "inside infiltrate": "#76a9e2",
}

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from spatial_tcr.stats import count_test

tcell_subtypes = ad_t.obs["cell_type_l2"].unique()
tcell_subtypes = ["gdT"]

scaler = 0.7 * 0.925

for t_subtype in tcell_subtypes:
    df = (
        ad_t.obs.groupby("tcell_infiltrate", observed=True)["cell_type_l2"]
        .value_counts()
        .unstack()
    )
    df["other T"] = df[[c for c in df.columns if c != t_subtype]].sum(axis=1)
    df = df[[t_subtype, "other T"]]
    df_norm = df.div(df.sum(axis=0), axis=1) * 100
    df_norm = df_norm.iloc[::-1]
    df_norm = df_norm.iloc[:, ::-1]
    print(df_norm)

    p_value = count_test(
        df.values[0, 0], df.values[0, 1], df.values[1, 0], df.values[1, 1]
    )

    # stacked barplot with p value
    fig, ax = plt.subplots(figsize=(1.5 * scaler, 5 * scaler))
    df_norm.T.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        width=0.6,
        color=[palette["outside infiltrate"], palette["inside infiltrate"]],
    )
    # Add absolute numbers inside each bar segment
    # Get the absolute counts (need to reverse rows and columns to match df_norm)
    df_abs = df.iloc[::-1, ::-1]

    # For stacked bars, patches are ordered: bar0_seg0, bar0_seg1, bar1_seg0, bar1_seg1, ...
    # Where bars correspond to rows of df_norm.T and segments correspond to columns of df_norm.T

    # Iterate through each bar (row in df_norm.T, which becomes a bar in the plot)
    for i, (_bar_name, _) in enumerate(df_norm.T.iterrows()):
        # Iterate through each segment (row in df_norm, which becomes a segment in the bar)
        for j, (_seg_name, _) in enumerate(df_norm.iterrows()):
            # Calculate patch index: (bar_index * num_segments) + segment_index
            rect_idx = i * len(df_norm) + j

            if rect_idx < len(ax.patches):
                rect = ax.patches[rect_idx]

                # Get the absolute count value
                abs_value = int(df_abs.iloc[j, i])
                # Get the percentage height
                height = df_norm.iloc[j, i]

                # Calculate text position (middle of the segment)
                # Get the bottom and top of the rectangle
                rect_bottom = rect.get_y()
                rect_height = rect.get_height()
                text_y = rect_bottom + rect_height / 2

                # Only add text if the segment is large enough to be visible
                if height > 2:  # Only show text if segment is > 2%
                    ax.text(
                        rect.get_x() + rect.get_width() / 2,
                        text_y,
                        str(abs_value),
                        ha="center",
                        va="center",
                        color="white" if height > 10 else "black",
                        fontweight="bold",
                        fontsize=6,
                    )

    if p_value < 0.001:
        stars = "***"
    elif p_value < 0.01:
        stars = "**"
    elif p_value < 0.05:
        stars = "*"
    else:
        stars = "ns"
    # Adjust the y-offset based on your data range:
    y_max = 100
    y = y_max + 0.05 * (y_max)  # starting height of the line
    h = 0.02 * (y_max)  # height of the significance bar

    x1, x2 = 0, 1
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=1.0, c="k")
    ax.text(
        (x1 + x2) * 0.5, y + h, stars, ha="center", va="bottom", color="k", fontsize=6
    )

    # Add a bit more space at the top of the plot for significance annotation
    ax.set_ylim(top=y + h + 0.05 * y_max, bottom=0)
    ax.set_ylabel("proportion [%]", fontsize=8)
    ax.set_xlabel("")
    ax.tick_params(axis="both", labelsize=6)

    # rotate x labels
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0, ha="center")

    # move legend outside and change title
    # Get the current legend handles
    handles = ax.get_legend().get_patches()
    ax.legend(
        handles,
        ["outside\ninfiltrate", "inside\ninfiltrate"],
        title="",
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        frameon=False,
        fontsize=6,
        title_fontsize=6,
    )

    sns.despine()
    plt.savefig(
        os.path.join(figure_dir, f"{t_subtype}_in_and_out_of_infiltrate.pdf"),
        dpi=300,
        bbox_inches="tight",
    )

## Run permutation test

In [ ]:
from spatial_tcr.clonal_expansion import calc_empirical_p_values

seed = 42
n_perms = 1000

tcell_subtypes = ad_t.obs["cell_type_l2"].unique()
tcell_subtypes = ["gdT"]

for t_subtype in tcell_subtypes:
    print(f"Analyzing {t_subtype}")

    ad_t.obs[f"{t_subtype}_vs_other"] = ad_t.obs["cell_type_l2"].astype(str)
    ad_t.obs.loc[
        ~ad_t.obs["cell_type_l2"].isin([t_subtype]), f"{t_subtype}_vs_other"
    ] = "other"
    rng = np.random.default_rng(seed)

    # null hypothesis: gdt cells are as likely to be inside the infiltrate as outside

    gdt_inside_obs = ad_t.obs.loc[
        ad_t.obs["tcell_infiltrate"] == "infiltrate", "cell_type_l2"
    ].value_counts()[t_subtype]

    gdt_inside_sim = []
    for _ in tqdm(range(n_perms)):
        gdt_inside_sim_sub = 0
        for sample in ad_t.obs["sample"].unique():
            ad_t_sub = ad_t[ad_t.obs["sample"] == sample]
            tmp_series = ad_t_sub.obs[f"{t_subtype}_vs_other"].copy()
            rng.shuffle(tmp_series.values)
            mask = ad_t_sub.obs["tcell_infiltrate"] == "infiltrate"

            # check if there are any gdT cells inside the infiltrate
            try:
                gdt_inside = tmp_series[mask].value_counts()[t_subtype]
            except Exception:
                gdt_inside = 0
            gdt_inside_sim_sub += gdt_inside

        gdt_inside_sim.append(gdt_inside_sim_sub)

    gdt_inside_sim = np.array(gdt_inside_sim)
    z_scores, p_values, q_values = calc_empirical_p_values(
        gdt_inside_obs, gdt_inside_sim, n_perms
    )

    print(q_values)

In [ ]:
n_gdts = ad_t.obs["cell_type_l2"].value_counts()["gdT"]
gdt_inside_sim_norm = (gdt_inside_sim / n_gdts) * 100
gdt_inside_obs_norm = (gdt_inside_obs / n_gdts) * 100

sns.set_theme(style="ticks", context="paper")
fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(gdt_inside_sim_norm, ax=ax)
ax.set_xlabel("fraction of gdT cells inside infiltrate [%]")
ax.set_ylabel("count")
ax.set_title("gdT cells inside infiltrate")

# draw red vertical line for observed number of clonal clusters
ax.axvline(gdt_inside_obs_norm, color="red", linestyle="--")
# add text for observed number of clonal clusters
ax.text(
    # gdt_inside_obs_norm + 1.5,
    gdt_inside_obs_norm + 0.3,
    # 700,
    120,
    f"observed fraction of gdT cells inside infiltrate: {gdt_inside_obs_norm:.2f}%\nempricial p-value: {np.round(p_values, 5)}",
    color="red",
    ha="left",
)

# split x axis into two parts
sns.despine(ax=ax)

plt.savefig(
    os.path.join(figure_dir_2, "gdT_inside_infiltrate_permutation_test.pdf"),
    dpi=300,
    bbox_inches="tight",
    transparent=True,
)

## Compare t cell density for gdT vs other T cells

In [ ]:
ad_t.obs["gdT_vs_other"] = ad_t.obs["cell_type_l2"].astype(str)
ad_t.obs.loc[~ad_t.obs["cell_type_l2"].isin(["gdT"]), "gdT_vs_other"] = "other"

In [ ]:
import seaborn as sns

sns.set_theme(style="ticks", context="paper")

sns.violinplot(
    data=ad_t.obs,
    x="gdT_vs_other",
    y="tcell_density",
)

## check minimal distance for each gdT cell to any other gdT cell and compare vs random t cells

In [ ]:
from sklearn.neighbors import NearestNeighbors

from spatial_tcr.spatial import spatial_sample_split

spatial_sample_split(ad_t, sample_key="sample")

distances_sub = []
for sample in ad_t.obs["sample"].unique():
    ad_t_sub = ad_t[ad_t.obs["sample"] == sample]

    gdt_coords = ad_t_sub[ad_t_sub.obs["cell_type_l2"] == "gdT"].obsm["spatial_split"]
    if len(gdt_coords) < 2:
        continue

    nn = NearestNeighbors(n_neighbors=2)
    nn.fit(gdt_coords)
    distances, indices = nn.kneighbors(gdt_coords)
    distances_sub.append(np.median(distances[:, 1]))

median_distance = np.median(distances_sub)
print(f"Median distance to nearest gdT: {median_distance:.2f}")

In [ ]:
# run permutation test for median distance to nearest gdT

seed = 42
n_perms = 1000

rng = np.random.default_rng(seed)

distances_sim = []

for _ in tqdm(range(n_perms)):
    distances_sub = []
    for sample in ad_t.obs["sample"].unique():
        ad_t_sub = ad_t[ad_t.obs["sample"] == sample]
        perm_idx = rng.shuffle(np.arange(ad_t_sub.shape[0]))
        tmp_values = ad_t_sub.obs["cell_type_l2"].astype(str).values
        rng.shuffle(tmp_values)

        gdt_coords = ad_t_sub[tmp_values == "gdT"].obsm["spatial_split"]
        if len(gdt_coords) < 2:
            continue
        nn = NearestNeighbors(n_neighbors=2)
        nn.fit(gdt_coords)
        distances, indices = nn.kneighbors(gdt_coords)
        distances_sub.append(np.median(distances[:, 1]))
    distances_sim.append(np.median(distances_sub))

distances_sim = np.array(distances_sim)
z_scores, p_values, q_values = calc_empirical_p_values(
    median_distance, distances_sim, n_perms
)

print(q_values)

In [ ]:
sns.set_theme(style="ticks", context="paper")
fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(distances_sim, ax=ax)
ax.set_xlabel("Median distance to nearest gdT")
ax.set_ylabel("Count")
ax.set_title("Median distance to nearest gdT")

# draw red vertical line for observed number of clonal clusters
ax.axvline(median_distance, color="red", linestyle="--")
# add text for observed number of clonal clusters
ax.text(
    median_distance - 1,
    105,
    f"Observed median distance to nearest gdT: {median_distance:.2f}\nEmpirical p-value: {np.round(q_values, 5)}",
    color="red",
    ha="right",
)

# split x axis into two parts
sns.despine(ax=ax)

## Prepare columns comining gdt glom and infiltrate

In [ ]:
sample = adata.obs["sample"].unique()[3]

ad_sub = adata[adata.obs["sample"] == sample].copy()

In [ ]:
ad_sub.obs.columns

In [ ]:
ad_sub.obs["gdT+glom"] = ad_sub.obs["in_glom"].astype(str)
mapping = {"False": "outside glom.", "True": "inside glom."}
ad_sub.obs["gdT+glom"] = ad_sub.obs["gdT+glom"].map(mapping)
ad_sub.obs.loc[ad_sub.obs["cell_type_l2"] == "gdT", "gdT+glom"] = "gdT"

# reorder categories
ad_sub.obs["gdT+glom"] = ad_sub.obs["gdT+glom"].astype("category")
ad_sub.obs["gdT+glom"] = ad_sub.obs["gdT+glom"].cat.reorder_categories(
    ["outside glom.", "inside glom.", "gdT"]
)

In [ ]:
ad_sub.obs["gdT+infiltrate"] = ad_sub.obs["tcell_infiltrate"].astype(str)
mapping = {"infiltrate": "T cell infiltrate", "no infiltrate": "other"}
ad_sub.obs["gdT+infiltrate"] = ad_sub.obs["gdT+infiltrate"].map(mapping)
ad_sub.obs.loc[ad_sub.obs["cell_type_l2"] == "gdT", "gdT+infiltrate"] = "gdT"

categories = ["other", "T cell infiltrate", "gdT"]

# reorder categories
ad_sub.obs["gdT+infiltrate"] = ad_sub.obs["gdT+infiltrate"].astype("category")
ad_sub.obs["gdT+infiltrate"] = ad_sub.obs["gdT+infiltrate"].cat.reorder_categories(
    categories
)

In [ ]:
ad_sub.obs["infiltrate+glom+gdT"] = ad_sub.obs["tcell_infiltrate"].astype(str)
mapping = {"infiltrate": "T cell infiltrate", "no infiltrate": "other"}
ad_sub.obs["infiltrate+glom+gdT"] = ad_sub.obs["infiltrate+glom+gdT"].map(mapping)

# add glom annotations
ad_sub.obs.loc[ad_sub.obs["in_glom"], "infiltrate+glom+gdT"] = "inside glom."

# add gdT annotations
ad_sub.obs.loc[ad_sub.obs["cell_type_l2"] == "gdT", "infiltrate+glom+gdT"] = "gdT"

categories = ["other", "T cell infiltrate", "inside glom.", "gdT"]

# reorder categories
ad_sub.obs["infiltrate+glom+gdT"] = ad_sub.obs["infiltrate+glom+gdT"].astype("category")
ad_sub.obs["infiltrate+glom+gdT"] = ad_sub.obs[
    "infiltrate+glom+gdT"
].cat.reorder_categories(categories)

## Run gdT neighborhood analysis

In [ ]:
adata.obs["gdT_vs_otherT"] = adata.obs["cell_type_l2"].astype(str)
t_mask = adata.obs["cell_type_l1"] == "T"
gdT_mask = adata.obs["cell_type_l2"] == "gdT"

adata.obs.loc[t_mask & ~gdT_mask, "gdT_vs_otherT"] = "other T"
adata.obs["gdT_vs_otherT"].value_counts()

In [ ]:
from spatial_tcr.nhood_analysis import run_nhood_comparison

ad_nhood_expr_merged, mean_comp, p_values = run_nhood_comparison(
    adata,
    group_1="gdT",
    group_2="other T",
    group_key="gdT_vs_otherT",
    comp_key="cell_type_l2",
    sample_key="sample",
    spatial_key="spatial",
    radius=25,
)

In [ ]:
mean_comp

In [ ]:
from spatial_tcr.colors import colors_dict, colors_new

palette = {
    "PC": "lightblue",
    "gdT": colors_new["gdT"],
    "DCT": colors_dict["DCT"],
    "PL": colors_dict["PL"],
}

# palette = {ct: colors_dict[ct] for ct in palette.keys()}
plot_utils.plot_palette_dict(palette)

In [ ]:
from spatial_tcr.nhood_analysis import compare_nhood_composition

compare_nhood_composition(
    mean_comp,
    title="Neighborhood cell type composition",
    topn=4,
    show_other=False,
    palette=palette,
    save_path=os.path.join(figure_dir, "gdT_nhood_composition.pdf"),
)

In [ ]:
def compare_specific_nhood_proportions(
    nhood_ct_comp_melted,
    ct,
    p_value: float,
    save_path: str = None,
):
    colors = sns.color_palette("Greys", n_colors=2)

    plot_df = nhood_ct_comp_melted[nhood_ct_comp_melted["cell_type"] == ct]

    plot_df.loc[:, "prop"] = plot_df["prop"] * 100

    fig, ax = plt.subplots(figsize=(1.5, 5))
    sns.barplot(
        x="group",
        y="prop",
        hue="group",
        data=plot_df,
        ax=ax,
        palette=colors,
        width=0.6,
    )
    ax.set_title(f"{ct}")
    ax.set_xlabel("")
    ax.set_ylabel("Proportion [%]")

    if p_value < 0.001:
        stars = "***"
    elif p_value < 0.01:
        stars = "**"
    elif p_value < 0.05:
        stars = "*"
    else:
        stars = "ns"
    # Adjust the y-offset based on your data range:
    y_max = np.max(plot_df["prop"].values)
    y = y_max + 0.05 * (y_max)  # starting height of the line
    h = 0.02 * (y_max)  # height of the significance bar

    x1, x2 = 0, 1
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=1.5, c="k")
    ax.text((x1 + x2) * 0.5, y + h, stars, ha="center", va="bottom", color="k")

    # Add a bit more space at the top of the plot for significance annotation
    ax.set_ylim(top=y + h + 0.05 * y_max, bottom=0)

    sns.despine()
    # plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
nhood_ct_comp_melted = pd.melt(
    mean_comp.reset_index(),
    id_vars="index",
    var_name="cell_type",
    value_name="proportion",
)
nhood_ct_comp_melted.columns = ["group", "cell_type", "prop"]

for ct in ["gdT", "PC", "DCT", "PL"]:
    compare_specific_nhood_proportions(
        nhood_ct_comp_melted.iloc[::-1],
        ct=ct,
        p_value=p_values[ct],
        save_path=os.path.join(figure_dir, f"{ct}_nhood_composition.pdf"),
    )